# Rebuild the Stance Statements

| Item | Description |
|---|---|
| Goal | Rebuild the text of every legislator statement on Topic 68, consumer data privacy legislation, from the saved speech-turn sentences, so the centroid (T3.5.3) and the policy summary (T3.5.4 to T3.5.7) start from one checked table. |
| Input | `track2_assigned.parquet` (one row per stance record with the cited sentence numbers), `member_turns.parquet` (the sentences of each legislator speech turn), and `hearings_sst.parquet` (hearing held dates), all under `/project2/mmiric_1435/acroh/llm_method/data`. |
| Output | `output/T3.5.2/topic68_statements.csv`: one row per statement with its text, stance, hearing year, and token count. |
| Unit | A statement: one stance record, that is, one legislator's extracted position on the topic. Its text is the turn sentences the extraction cited, in order, joined by a space. A speech turn may hold several statements. |
| Sample | All Topic 68 stance records in the saved extraction (374, hearings held 1999–2025). |
| Variables | • `text`: the statement's cited sentences joined by a space.<br>• `hearing_year`: the calendar year of the hearing's held date.<br>• `n_tokens`: tokens of `text` under the `bge-large-en-v1.5` tokenizer, including its start and end tokens; must be at most 512 to embed whole. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for how this table is used downstream.


## Step 1. Import packages

Read the saved tables, count tokens with the encoder's tokenizer, and display checks.


In [1]:
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer
from IPython.display import display


## Step 2. Define file locations

Read the source files and the tokenizer in place; save the output in `output/T3.5.2/` under the TEMP folder.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SOURCE_DATA_FOLDER = Path("/project2/mmiric_1435/acroh/llm_method/data")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.5.2"  # the identifier names this notebook's output folder
RECORDS_FILE = SOURCE_DATA_FOLDER / "interim/topics/taxonomy/track2_assigned.parquet"
TURNS_FILE = SOURCE_DATA_FOLDER / "interim/topics/track2/member_turns.parquet"
HEARINGS_FILE = SOURCE_DATA_FOLDER / "interim/repo_data/raw/02_hearings/hearings_sst.parquet"
MODEL_FOLDER = Path(
    "/scratch2/acroh/projects/llm_method/hf_cache/hub/models--BAAI--bge-large-en-v1.5"
    "/snapshots/d4aa6901d3a41ba39fb536a557fa166f842b0e09"
)
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

TOPIC_ID = 68  # consumer data privacy legislation
MAX_TOKENS = 512  # the encoder's input limit
TURN_KEYS = ["packageId", "turn_idx"]
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.2


## Step 3. Load the Topic 68 stance records

Each record is one statement with a support or concern label and the numbers of the turn sentences it cites. Sentence numbers start at 1.


In [3]:
statements = pd.read_parquet(RECORDS_FILE, filters=[("topic_id", "==", TOPIC_ID)])
statements = statements.rename(columns={"sentences": "sentence_numbers"})
assert statements["stance"].isin(["support", "concern"]).all()
assert statements[TURN_KEYS + ["bioGuideId", "record_idx"]].notna().all().all()
assert not statements.duplicated(TURN_KEYS + ["record_idx"]).any()
print(f"Topic {TOPIC_ID} statements: {len(statements):,}; label: {statements['topic_label'].iloc[0]}")
print(f"By stance: {statements['stance'].value_counts().to_dict()}")
display(statements[TURN_KEYS + ["record_idx", "bioGuideId", "stance", "sentence_numbers"]].head(3))


Topic 68 statements: 374; label: consumer data privacy legislation
By stance: {'support': 221, 'concern': 153}


,packageId,turn_idx,record_idx,bioGuideId,stance,sentence_numbers
0,CHRG-106shrg69984,4,1,K000148,support,[9]
1,CHRG-106shrg69984,9,7,A000356,support,[46]
2,CHRG-106shrg69984,9,8,A000356,support,[48]


## Step 4. Load the speech turns of those hearings

Read only the turns from hearings that contain a Topic 68 statement. Each turn carries its sentences as a list.


In [4]:
turns = pd.read_parquet(
    TURNS_FILE,
    columns=TURN_KEYS + ["bioGuideId", "sentences"],
    filters=[("packageId", "in", statements["packageId"].unique().tolist())],
)
turns = turns.rename(columns={"bioGuideId": "turn_bioGuideId", "sentences": "turn_sentences"})
assert not turns.duplicated(TURN_KEYS).any()
print(f"Speech turns loaded: {len(turns):,} from {turns['packageId'].nunique():,} hearings")
display(turns[TURN_KEYS + ["turn_bioGuideId"]].head(3))


Speech turns loaded: 16,288 from 119 hearings


,packageId,turn_idx,turn_bioGuideId
0,CHRG-106shrg69984,1,B001126
1,CHRG-106shrg69984,2,K000148
2,CHRG-106shrg69984,3,B001126


## Step 5. Attach each statement's turn

A many-to-one join gives every statement the sentence list of its turn. Every statement must find its turn, and the speaker recorded on the statement must match the turn's speaker.


In [5]:
statements = statements.merge(turns, on=TURN_KEYS, how="left", validate="many_to_one")
assert statements["turn_sentences"].notna().all()
assert statements["bioGuideId"].eq(statements["turn_bioGuideId"]).all()
print(f"Statements matched to a turn: {len(statements):,}")
display(statements[TURN_KEYS + ["record_idx", "stance", "sentence_numbers"]].head(3))


Statements matched to a turn: 374


,packageId,turn_idx,record_idx,stance,sentence_numbers
0,CHRG-106shrg69984,4,1,support,[9]
1,CHRG-106shrg69984,9,7,support,[46]
2,CHRG-106shrg69984,9,8,support,[48]


## Step 6. Rebuild each statement's text

Take the cited sentences from the turn, in the order cited, and join them with a space. Every cited number must fall inside the turn, and no statement may come out empty.


In [6]:
texts = []
for sentence_numbers, turn_sentences in zip(statements["sentence_numbers"], statements["turn_sentences"]):
    sentence_numbers = [int(number) for number in sentence_numbers]
    assert sentence_numbers
    assert min(sentence_numbers) >= 1 and max(sentence_numbers) <= len(turn_sentences)
    cited_sentences = [turn_sentences[number - 1] for number in sentence_numbers]
    texts.append(" ".join(cited_sentences))
statements["text"] = texts
statements["n_sentences"] = statements["sentence_numbers"].map(len)
assert statements["text"].str.strip().ne("").all()
print(f"Statements with text: {len(statements):,}")
print(f"Sentences per statement: min {statements['n_sentences'].min()}, median {int(statements['n_sentences'].median())}, max {statements['n_sentences'].max()}")
for example in statements.sample(3, random_state=0).itertuples(index=False):
    print(f"\n[{example.stance}] {example.text}")


Statements with text: 374
Sentences per statement: min 1, median 1, max 8

[concern] I do have concerns with the risks posed by the EU's General Data Protection Regulations and the California Consumer Privacy Act. Those concern what it will mean to the Internet ecosystem, especially the innovative entrepreneurial businesses that are positioned to be disadvantaged based upon these regulatory overhauls and in some instances these regulations conflict one with another.

[support] Passage of Federal data privacy legislation is long overdue.

[concern] There has to be a way that you can figure out how to do what's necessary-- because we all see people in the airports going through with long skirts and lots of cover-up possibilities, and that's what you're trying to address, and we understand that--there has got to be a way; however, for a privacy concern to be addressed, because it's a legitimate concern.


## Step 7. Attach the hearing year

Use the hearing's held date, as the T2.4 figures do, rather than the record's own year field.


In [7]:
hearings = pd.read_parquet(HEARINGS_FILE, columns=["packageId", "heldDate_first"])
assert hearings["packageId"].is_unique
statements = statements.merge(hearings, on="packageId", how="left", validate="many_to_one")
assert statements["heldDate_first"].notna().all()
statements["hearing_year"] = pd.to_datetime(statements["heldDate_first"]).dt.year
print(f"Hearing years: {statements['hearing_year'].min()}–{statements['hearing_year'].max()}; hearings: {statements['packageId'].nunique():,}")


Hearing years: 1999–2025; hearings: 119


## Step 8. Count the tokens of each statement

Every statement must fit the encoder's 512-token limit, otherwise its embedding in T3.5.3 would be cut off silently.


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_FOLDER, local_files_only=True)
statements["n_tokens"] = [len(tokenizer(text)["input_ids"]) for text in statements["text"]]
assert statements["n_tokens"].le(MAX_TOKENS).all()
print(f"Tokens per statement: min {statements['n_tokens'].min()}, median {int(statements['n_tokens'].median())}, max {statements['n_tokens'].max()}")
print(f"Total tokens across all statements: {statements['n_tokens'].sum():,}")


Tokens per statement: min 8, median 38, max 245
Total tokens across all statements: 17,831


## Step 9. Count statements by stance and hearing year

A coverage table for the record.


In [9]:
coverage = pd.crosstab(statements["hearing_year"], statements["stance"])
coverage["total"] = coverage.sum(axis=1)
assert coverage["total"].sum() == len(statements)
display(coverage.T)


hearing_year,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
stance,,,,,,,,,,,,,,,,,,,,,
concern,5,15,7,6,5,2,1,5,1,3,...,11,6,15,9,1,21,0,2,2,3
support,4,18,8,7,3,2,1,3,0,3,...,10,5,29,31,5,41,1,8,7,3
total,9,33,15,13,8,4,2,8,1,6,...,21,11,44,40,6,62,1,10,9,6


## Step 10. Save the statement table

Save one row per statement in hearing order. Later notebooks read this file and never go back to the turn sentences.


In [10]:
output_columns = [
    "packageId", "turn_idx", "record_idx", "bioGuideId", "hearing_year", "stance", "text", "n_tokens",
]
statements = statements.sort_values(["hearing_year", "packageId", "turn_idx", "record_idx"])
statements_file = OUTPUT_FOLDER / "topic68_statements.csv"
statements[output_columns].to_csv(statements_file, index=False)
print(f"Saved: {statements_file}")
print(f"Rows: {len(statements):,}; columns: {output_columns}")
display(statements[output_columns].head(3))


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.2/topic68_statements.csv
Rows: 374; columns: ['packageId', 'turn_idx', 'record_idx', 'bioGuideId', 'hearing_year', 'stance', 'text', 'n_tokens']


,packageId,turn_idx,record_idx,bioGuideId,hearing_year,stance,text,n_tokens
0,CHRG-106shrg69984,4,1,K000148,1999,support,The bill is an important first step that recog...,39
1,CHRG-106shrg69984,9,7,A000356,1999,support,"With all due respect, this process is too long...",48
2,CHRG-106shrg69984,9,8,A000356,1999,support,The second area where I think the PROTECT Act ...,48
